# Data Drift Guardian — демонстрация

Полный цикл работы системы на демонстрационном датасете кредитного скоринга с искусственно внесённым дрейфом:
генерация данных → анализ (`analyze`) → интерпретация метрик → графики → экспорт HTML/JSON.

Все данные генерируются с фиксированным seed, поэтому результаты воспроизводимы.

In [1]:
import sys
sys.path.insert(0, "..")  # запуск из папки notebooks без установки пакета

import pandas as pd
from drift_guardian import DriftConfig, analyze
from drift_guardian.demo import SCENARIOS, make_demo
from drift_guardian.plots import column_summary_frame, style_severity, numeric_distribution_figure, categorical_distribution_figure
from drift_guardian.html_report import save_html_report, report_to_json

pd.set_option("display.width", 160)
for name, description in SCENARIOS.items():
    print(f"{name:14s} — {description}")

no_drift       — Без дрейфа — контрольный батч из того же распределения
mean_shift     — Сдвиг среднего — аудитория постарела, доходы выросли на 25%
missing_surge  — Рост пропусков — доля пустого дохода выросла на ~11 п.п.
new_category   — Новая категория — появился канал «партнёрская сеть»
concept_drift  — Концептуальный дрейф — признаки стабильны, но доля дефолтов выросла с ~16% до ~40%
mixed          — Всё вместе — плюс рост разброса сумм кредита


## 1. Данные

Эталон — 20 000 заявок на кредит, на которых «обучалась модель». Текущий батч — 5 000 заявок из продакшна.
Колонка `target` (1 — дефолт) порождена логистической моделью от признаков, поэтому в сценарии
`concept_drift` меняется связь «признаки → дефолт», а сами признаки — нет.

In [2]:
reference, current = make_demo("mixed", ref_rows=20_000, cur_rows=5_000, seed=42)
print(reference.shape, current.shape)
reference.head()

(20000, 9) (5000, 9)


,age,income,loan_amount,credit_history_years,num_dependents,employment_type,region,channel,target
0,21.0,69728.0,203308.0,4.3,3,наёмный,Москва,мобильное приложение,0
1,47.0,35597.0,57049.0,19.2,1,наёмный,Миллионники,офис,0
2,37.0,87136.0,45884.0,10.6,1,наёмный,Москва,мобильное приложение,1
3,32.0,95825.0,204904.0,7.3,3,наёмный,Москва,мобильное приложение,0
4,35.0,22572.0,18076.0,6.0,3,наёмный,Москва,мобильное приложение,1


## 2. Сценарий «всё вместе» (`mixed`)

Указываем целевую переменную — она анализируется отдельным блоком и не участвует в adversarial validation.

In [3]:
config = DriftConfig(target_column="target")
report = analyze(reference, current, config)

print("Статус:", report["overall_severity"].upper())
print("Рекомендация:", report["recommendation"])
print()
for alert in report["alerts"]:
    print("-", alert)

Статус: CRITICAL
Рекомендация: Зафиксирован критический дрейф — рекомендуется переобучение модели.

- ВНИМАНИЕ: Доля пропусков в 'income' выросла с 7.7% до 19.3% (+11.6 п.п.).
- КРИТИЧНО: 16.8% значений 'loan_amount' вне диапазона эталона [5857, 1.23218e+06].
- КРИТИЧНО: В 'channel' появились категории, не предусмотренные эталоном: ['партнёрская сеть'] — 11.0% строк текущего батча.
- ВНИМАНИЕ: дрейф целевой переменной 'target' — chi2=429 (p=<1e-16), jensen_shannon=0.13.
- КРИТИЧНО: дрейф по признаку 'age' — ks=0.239 (p=<1e-16), psi=0.337, jensen_shannon=0.243, wasserstein_norm=0.605.
- ВНИМАНИЕ: дрейф по признаку 'income' — ks=0.185 (p=<1e-16), psi=0.193, jensen_shannon=0.185, wasserstein_norm=0.465.
- КРИТИЧНО: дрейф по признаку 'loan_amount' — ks=0.221 (p=<1e-16), psi=0.388, jensen_shannon=0.26, wasserstein_norm=0.428.
- КРИТИЧНО: дрейф по признаку 'channel' — chi2=2.25e+03 (p=<1e-16), psi=0.782, jensen_shannon=0.238.
- КРИТИЧНО: adversarial validation различает выборки (ROC-AUC=0.84

### Сводка по признакам

Худшие признаки сверху. `KS / χ²` — статистика p-value-теста, `PSI`, `JS`, `Вассерштейн` — метрики размера эффекта.

In [4]:
style_severity(column_summary_frame(report))

,признак,тип,статус,PSI,JS,Вассерштейн (норм.),KS / χ²,p-value
0,channel,категориальный,🛑 critical,0.781800,0.238400,nan,2248.780000,0.000000
1,loan_amount,числовой,🛑 critical,0.388000,0.260000,0.427600,0.221300,0.000000
2,age,числовой,🛑 critical,0.336800,0.242700,0.604900,0.238900,0.000000
3,income,числовой,⚠️ warning,0.193100,0.184800,0.465200,0.185500,0.000000
4,credit_history_years,числовой,✅ ok,0.000900,0.012800,0.012900,0.008800,0.909902
5,employment_type,категориальный,✅ ok,0.000400,0.008600,nan,1.652600,0.647514
6,num_dependents,категориальный,✅ ok,0.000300,0.007800,nan,1.357100,0.715621
7,region,категориальный,✅ ok,0.000200,0.006600,nan,0.961700,0.810527


### Распределения

Синий — эталон, оранжевый — текущий батч. У `age` виден сдвиг вправо (аудитория постарела),
у `channel` — новая категория «партнёрская сеть».

In [5]:
numeric_distribution_figure(reference["age"], current["age"], title="age").show()
categorical_distribution_figure(reference["channel"], current["channel"], title="channel").show()

### Adversarial validation

LightGBM учится отличать эталон от батча. ROC-AUC ≈ 0.5 — выборки неразличимы; чем выше — тем сильнее
изменилась совместная структура признаков. Важности показывают, что именно изменилось.

In [6]:
adv = report["adversarial"]
print(f"ROC-AUC = {adv['roc_auc']:.3f} ({adv['severity']}), бэкенд: {adv['backend']}")
pd.DataFrame(adv["top_features"])

ROC-AUC = 0.845 (critical), бэкенд: lightgbm


,feature,importance
0,loan_amount,0.4220
1,income,0.2307
2,channel,0.1532
3,age,0.1431
4,credit_history_years,0.0324
5,num_dependents,0.0063
6,employment_type,0.0062
7,region,0.0060


## 3. Все сценарии

Один и тот же эталон, разные батчи. Обратите внимание на `no_drift` (ничего не найдено) и `concept_drift`
(признаки стабильны, но целевая переменная изменилась).

In [7]:
rows = []
for name in SCENARIOS:
    ref, cur = make_demo(name, 20_000, 5_000, seed=42)
    r = analyze(ref, cur, config)
    rows.append({
        "сценарий": name,
        "итог": r["overall_severity"],
        "таргет": r["target_drift"]["severity"],
        "критичных признаков": sum(c["severity"] == "critical" for c in r["columns"]),
        "DQ-алертов": len(r["data_quality"]),
        "adversarial AUC": r["adversarial"]["roc_auc"],
    })
pd.DataFrame(rows)

,сценарий,итог,таргет,критичных признаков,DQ-алертов,adversarial AUC
0,no_drift,ok,ok,0,0,0.5030
1,mean_shift,critical,ok,1,0,0.7561
2,missing_surge,warning,ok,0,1,0.5532
3,new_category,critical,ok,1,1,0.5627
4,concept_drift,critical,critical,0,0,0.5204
5,mixed,critical,warning,3,3,0.8451


## 4. Концептуальный дрейф крупным планом

В сценарии `concept_drift` входные данные не изменились, но доля дефолтов при тех же заявках выросла
с ~16 % до ~40 %. Поколоночные тесты признаков молчат, adversarial validation молчит, а блок целевой переменной
поднимает тревогу — именно этот случай был бы невидим для мониторинга «только по X».

In [8]:
ref_c, cur_c = make_demo("concept_drift", 20_000, 5_000, seed=42)
r = analyze(ref_c, cur_c, config)
print("Итог:", r["overall_severity"], "| adversarial AUC:", r["adversarial"]["roc_auc"])
print("Рекомендация:", r["recommendation"])
print("Доля дефолтов: эталон", round(ref_c["target"].mean(), 3), "→ батч", round(cur_c["target"].mean(), 3))
pd.DataFrame(r["target_drift"]["tests"])[["name", "statistic", "p_value", "severity"]]

Итог: critical | adversarial AUC: 0.5204
Рекомендация: Признаки стабильны, но распределение целевой переменной изменилось — вероятен концептуальный дрейф. Рекомендуется переобучение модели на свежих размеченных данных.
Доля дефолтов: эталон 0.154 → батч 0.402


,name,statistic,p_value,severity
0,chi2,1528.5269,0.0,warning
1,psi,0.3265,NaN,critical
2,jensen_shannon,0.2392,NaN,critical


## 5. Пороги и ложные срабатывания

p-value-тесты (KS, χ²) на больших выборках находят значимость почти всегда, а на одинаковых распределениях
ложно срабатывают с частотой α. Поэтому в системе действует двухключевое правило: p-value-тест засчитывается,
только если метрики размера эффекта тоже превысили порог. Проверим на батче без дрейфа, как часто KS был бы
«значим» без этого правила.

In [9]:
ref_n, cur_n = make_demo("no_drift", 20_000, 5_000, seed=7)
r = analyze(ref_n, cur_n, config)
significant = [c["column"] for c in r["columns"] if any(t.get("details", {}).get("significant") for t in c["tests"])]
print("Итог по двухключевому правилу:", r["overall_severity"])
print("Колонки, где p-value-тест был статистически значим:", significant or "нет")

Итог по двухключевому правилу: ok
Колонки, где p-value-тест был статистически значим: нет


## 6. Экспорт

HTML-отчёт с интерактивными графиками и JSON по выходному контракту.

In [10]:
from pathlib import Path
out = Path("output"); out.mkdir(exist_ok=True)
path = save_html_report(out / "drift_report_mixed.html", report, reference, current, plotlyjs="cdn")
(out / "drift_report_mixed.json").write_text(report_to_json(report), encoding="utf-8")
print("HTML:", path, f"({path.stat().st_size // 1024} КБ)")
print("Ключи JSON:", list(report.keys()))

HTML: output/drift_report_mixed.html (99 КБ)
Ключи JSON: ['overall_severity', 'recommendation', 'alerts', 'schema', 'data_quality', 'columns', 'adversarial', 'meta', 'target_drift', 'prediction_drift']


## Выводы

- Система различает четыре класса проблем: дрейф признаков, проблемы качества данных, многомерный сдвиг
  (adversarial validation) и концептуальный дрейф по целевой переменной.
- Пороги метрик размера эффекта откалиброваны друг относительно друга, а p-value-тесты не шумят на больших данных
  благодаря двухключевому правилу.
- Тот же анализ доступен из дашборда Streamlit (`streamlit run app/streamlit_app.py`) и из командной строки
  (`drift-guardian -r reference.csv -c current.csv --target target --html report.html`).